# JARVIS Doc Search — Agent de documentation
Recherche full-text dans scripts/, core/, notebooks/, config/

In [ ]:
import subprocess, json, re
from pathlib import Path

JARVIS_ROOT = Path.home() / 'IA/Core/jarvis'
SEARCH_DIRS = ['scripts', 'core', 'notebooks', 'config', 'orchestrator']
EXTENSIONS = ['.py', '.md', '.sh', '.json', '.ipynb']

def index_files():
    files = []
    for d in SEARCH_DIRS:
        for ext in EXTENSIONS:
            files.extend((JARVIS_ROOT / d).rglob(f'*{ext}'))
    return sorted(set(files))

all_files = index_files()
print(f'Index: {len(all_files)} fichiers dans {JARVIS_ROOT}')

In [ ]:
def search(query: str, max_results: int = 20) -> list[dict]:
    """Grep full-text dans tous les fichiers indexés."""
    pattern = re.compile(query, re.IGNORECASE)
    results = []
    for f in all_files:
        try:
            text = f.read_text(errors='ignore')
            matches = []
            for i, line in enumerate(text.splitlines(), 1):
                if pattern.search(line):
                    matches.append({'line': i, 'content': line.strip()[:120]})
            if matches:
                results.append({'file': str(f.relative_to(JARVIS_ROOT)), 'matches': matches[:3]})
        except Exception:
            pass
    return results[:max_results]

# Exemple de recherche
results = search('lm_guard')
for r in results:
    print(f"\n📄 {r['file']}")
    for m in r['matches']:
        print(f"  L{m['line']}: {m['content']}")

In [ ]:
# Recherche interactive
QUERY = 'sync_config'  # ← modifie ici
results = search(QUERY)
print(f'\n🔍 "{QUERY}" → {len(results)} fichiers')
for r in results:
    print(f"  {r['file']} ({len(r['matches'])} occurrences)")

In [ ]:
# Interroger le LLM local sur la doc
def ask_lm(question: str, context_query: str = None) -> str:
    context = ''
    if context_query:
        hits = search(context_query, max_results=5)
        ctx_lines = []
        for r in hits:
            ctx_lines.append(f"# {r['file']}")
            for m in r['matches']:
                ctx_lines.append(f"  L{m['line']}: {m['content']}")
        context = '\n'.join(ctx_lines)
    
    prompt = f"{context}\n\n{question}" if context else question
    result = subprocess.run(
        ['bash', str(Path.home()/'jarvis/scripts/lm-ask.sh'), prompt],
        capture_output=True, text=True, timeout=60
    )
    return result.stdout.strip() or result.stderr.strip()

# Exemple
answer = ask_lm('Comment fonctionne sync_config.py en mode daemon ?', context_query='sync_config')
print(answer)

In [ ]:
# Vue arborescence
from collections import defaultdict
tree = defaultdict(list)
for f in all_files:
    rel = f.relative_to(JARVIS_ROOT)
    tree[str(rel.parent)].append(rel.name)

for folder in sorted(tree):
    files_in = sorted(tree[folder])
    print(f"\n📁 {folder}/  ({len(files_in)} fichiers)")
    for fn in files_in[:10]:
        print(f"   {fn}")
    if len(files_in) > 10:
        print(f"   ... +{len(files_in)-10} autres")